# NBA Dataset Updater

This notebook updates the NBA dataset with the latest data from the NBA API.
It can be run daily to keep the dataset current with new games and statistics.

## What this notebook does:
1. Checks the most recent date in your existing dataset
2. Fetches new games and statistics since that date
3. Updates the following CSV files:
   - Games.csv
   - PlayerStatistics.csv
   - TeamStatistics.csv
   - Players.csv (new players)
4. Preserves the exact schema of your existing files
5. Only makes changes if there is new data available

## Install Required Packages

In [1]:
!pip install nba_api pandas

## Import Libraries

In [2]:
import pandas as pd
import os
from datetime import datetime, timedelta
from nba_api.stats.endpoints import leaguegamefinder, boxscoretraditionalv3, commonallplayers
from nba_api.stats.static import teams
import time
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print("Using BoxScoreTraditionalV3 (V3 endpoint)")

Libraries imported successfully!
Using BoxScoreTraditionalV3 (V3 endpoint)


## Configuration

In [3]:
# Set the base path for your dataset
BASE_PATH = "nba-stats/"

# Define file paths
GAMES_FILE = os.path.join(BASE_PATH, "Games.csv")
PLAYERS_FILE = os.path.join(BASE_PATH, "Players.csv")
PLAYER_STATS_FILE = os.path.join(BASE_PATH, "PlayerStatistics.csv")
TEAM_STATS_FILE = os.path.join(BASE_PATH, "TeamStatistics.csv")

# API rate limiting (to avoid hitting NBA API rate limits)
API_DELAY = 0.6  # seconds between API calls

print(f"Dataset path: {BASE_PATH}")
print(f"Games file: {GAMES_FILE}")
print(f"Players file: {PLAYERS_FILE}")
print(f"Player stats file: {PLAYER_STATS_FILE}")
print(f"Team stats file: {TEAM_STATS_FILE}")

Dataset path: nba-stats/
Games file: nba-stats/Games.csv
Players file: nba-stats/Players.csv
Player stats file: nba-stats/PlayerStatistics.csv
Team stats file: nba-stats/TeamStatistics.csv


## Load Existing Data

In [4]:
# Load existing CSV files
print("Loading existing dataset...")

games_df = pd.read_csv(GAMES_FILE)
players_df = pd.read_csv(PLAYERS_FILE)
player_stats_df = pd.read_csv(PLAYER_STATS_FILE)
team_stats_df = pd.read_csv(TEAM_STATS_FILE)

print(f"✓ Loaded {len(games_df)} games")
print(f"✓ Loaded {len(players_df)} players")
print(f"✓ Loaded {len(player_stats_df)} player statistics records")
print(f"✓ Loaded {len(team_stats_df)} team statistics records")

# Find the most recent game date in the dataset
games_df['gameDate'] = pd.to_datetime(games_df['gameDate'], format='mixed', utc=True)
most_recent_date = games_df['gameDate'].max()

print(f"\nMost recent game in dataset: {most_recent_date}")
print(f"Today's date: {datetime.now().strftime('%Y-%m-%d')}")

Loading existing dataset...
✓ Loaded 72176 games
✓ Loaded 6679 players
✓ Loaded 1636001 player statistics records
✓ Loaded 144368 team statistics records

Most recent game in dataset: 2025-11-20 00:00:00+00:00
Today's date: 2025-12-15


## Check for New Games

In [5]:
def get_new_games(since_date):
    """
    Fetch all games that occurred after the given date.
    """
    print(f"\nFetching games since {since_date.strftime('%Y-%m-%d')}...")
    
    try:
        # Get all games from the league game finder
        # We'll use the current season and filter by date
        print(f"  API call with date: {since_date.strftime('%m/%d/%Y')}")
        
        gamefinder = leaguegamefinder.LeagueGameFinder(
            date_from_nullable=since_date.strftime('%m/%d/%Y'),
            league_id_nullable='00'  # NBA
        )
        
        print("  Extracting data from API response...")
        games = gamefinder.get_data_frames()[0]
        
        if len(games) == 0:
            print("  No new games found.")
            return pd.DataFrame()
        
        # The API returns two rows per game (one for each team)
        # Group by GAME_ID to get unique games
        unique_game_ids = games['GAME_ID'].unique()
        
        print(f"  Found {len(unique_game_ids)} new games")
        print(f"  Total rows returned (2 per game): {len(games)}")
        
        # Show sample of what was found
        if len(games) > 0:
            print(f"  Sample game dates: {games['GAME_DATE'].unique()[:5]}")
        
        return games
        
    except Exception as e:
        print(f"❌ Error fetching games: {e}")
        print(f"   Error type: {type(e).__name__}")
        import traceback
        print(f"   Traceback:")
        traceback.print_exc()
        return pd.DataFrame()

# Calculate the date to start fetching from (day after most recent game)
fetch_from_date = most_recent_date + timedelta(days=1)

print(f"Most recent game date: {most_recent_date}")
print(f"Fetching games from: {fetch_from_date}")

# Get new games
new_games_raw = get_new_games(fetch_from_date)

if len(new_games_raw) > 0:
    print(f"\n✅ New data available! Will process {len(new_games_raw['GAME_ID'].unique())} games.")
    print(f"   Date range: {new_games_raw['GAME_DATE'].min()} to {new_games_raw['GAME_DATE'].max()}")
else:
    print("\n✓ Dataset is up to date! No new games to process.")

Most recent game date: 2025-11-20 00:00:00+00:00
Fetching games from: 2025-11-21 00:00:00+00:00

Fetching games since 2025-11-21...
  API call with date: 11/21/2025
  Extracting data from API response...
  Found 156 new games
  Total rows returned (2 per game): 312
  Sample game dates: ['2025-12-14' '2025-12-13' '2025-12-12' '2025-12-11' '2025-12-10']

✅ New data available! Will process 156 games.
   Date range: 2025-11-21 to 2025-12-14


## Process New Games

In [6]:
def process_games_data(games_raw_df, existing_games_df):
    """
    Process raw game data from API into the Games.csv format.
    """
    if len(games_raw_df) == 0:
        return pd.DataFrame()
    
    print("\nProcessing games data...")
    
    # Group by game to get one row per game
    new_games_list = []
    
    for game_id in games_raw_df['GAME_ID'].unique():
        game_data = games_raw_df[games_raw_df['GAME_ID'] == game_id]
        
        # Skip if game already exists
        if game_id in existing_games_df['gameId'].values:
            continue
        
        # Identify home and away teams
        home_team = game_data[game_data['MATCHUP'].str.contains('vs.')].iloc[0] if len(game_data[game_data['MATCHUP'].str.contains('vs.')]) > 0 else None
        away_team = game_data[game_data['MATCHUP'].str.contains('@')].iloc[0] if len(game_data[game_data['MATCHUP'].str.contains('@')]) > 0 else None
        
        if home_team is None or away_team is None:
            # Fallback: use WL to determine winner and team info
            teams_in_game = game_data.sort_values('TEAM_ID')
            if len(teams_in_game) >= 2:
                home_team = teams_in_game.iloc[0]
                away_team = teams_in_game.iloc[1]
        
        # Determine winner
        if home_team['WL'] == 'W':
            winner = home_team['TEAM_ID']
        else:
            winner = away_team['TEAM_ID']
        
        # Convert game date to proper format (ISO 8601 with UTC)
        game_date_str = home_team['GAME_DATE']
        game_date_obj = pd.to_datetime(game_date_str)
        game_date_formatted = game_date_obj.strftime('%Y-%m-%dT%H:%M:%SZ')
        
        # Create game record matching the schema
        game_record = {
            'gameId': game_id,
            'gameDate': game_date_formatted,
            'hometeamCity': home_team['TEAM_NAME'].rsplit(' ', 1)[0] if ' ' in home_team['TEAM_NAME'] else home_team['TEAM_NAME'],
            'hometeamName': home_team['TEAM_NAME'].rsplit(' ', 1)[-1] if ' ' in home_team['TEAM_NAME'] else '',
            'hometeamId': home_team['TEAM_ID'],
            'awayteamCity': away_team['TEAM_NAME'].rsplit(' ', 1)[0] if ' ' in away_team['TEAM_NAME'] else away_team['TEAM_NAME'],
            'awayteamName': away_team['TEAM_NAME'].rsplit(' ', 1)[-1] if ' ' in away_team['TEAM_NAME'] else '',
            'awayteamId': away_team['TEAM_ID'],
            'homeScore': home_team['PTS'],
            'awayScore': away_team['PTS'],
            'winner': winner,
            'gameType': '',
            'attendance': '',
            'arenaId': '',
            'gameLabel': '',
            'gameSubLabel': '',
            'seriesGameNumber': ''
        }
        
        new_games_list.append(game_record)
    
    if len(new_games_list) > 0:
        new_games_df = pd.DataFrame(new_games_list)
        print(f"✓ Processed {len(new_games_df)} new games")
        return new_games_df
    else:
        print("No new unique games to add")
        return pd.DataFrame()

# Process the new games
new_games_df = process_games_data(new_games_raw, games_df)


Processing games data...
✓ Processed 156 new games


## Validate Existing Data and Identify Missing Statistics

In [7]:
def find_games_missing_stats():
    """
    Identify all games that are missing player or team statistics.
    Returns a list of game IDs that need box scores fetched.
    """
    print("\n" + "="*50)
    print("CHECKING FOR MISSING STATISTICS")
    print("="*50)
    
    # CRITICAL: Normalize game IDs by removing leading zeros for consistent comparison
    # Games.csv may have mixed format (some with leading zeros, some without)
    def normalize_game_id(gid):
        """Remove leading zeros from game ID for consistent comparison"""
        return str(gid).lstrip('0') if pd.notna(gid) else gid
    
    # Get unique game IDs from each dataset and normalize them
    all_game_ids = set(games_df['gameId'].apply(normalize_game_id).unique())
    games_with_player_stats = set(player_stats_df['gameId'].apply(normalize_game_id).unique())
    games_with_team_stats = set(team_stats_df['gameId'].apply(normalize_game_id).unique())
    
    # Find games missing statistics
    games_missing_player_stats = all_game_ids - games_with_player_stats
    games_missing_team_stats = all_game_ids - games_with_team_stats
    
    # Combine all games that need data (union of both sets)
    games_needing_data = games_missing_player_stats | games_missing_team_stats
    
    print(f"\nTotal games in Games.csv: {len(all_game_ids)}")
    print(f"Games with player statistics: {len(games_with_player_stats)}")
    print(f"Games with team statistics: {len(games_with_team_stats)}")
    print(f"\nGames missing player stats: {len(games_missing_player_stats)}")
    print(f"Games missing team stats: {len(games_missing_team_stats)}")
    print(f"Total games needing data: {len(games_needing_data)}")
    
    if games_needing_data:
        print(f"\n⚠️  Found {len(games_needing_data)} games that need statistics!")
        
        # Show sample of missing games
        # Create normalized column for matching
        games_df_normalized = games_df.copy()
        games_df_normalized['gameId_normalized'] = games_df_normalized['gameId'].apply(normalize_game_id)
        
        missing_games_df = games_df_normalized[games_df_normalized['gameId_normalized'].isin(games_needing_data)].sort_values('gameDate', ascending=False)
        
        print("\nSample of games missing statistics (most recent first):")
        for idx, game in missing_games_df.head(10).iterrows():
            print(f"  - {game['gameId']}: {game['awayteamCity']} {game['awayteamName']} @ {game['hometeamCity']} {game['hometeamName']} ({game['gameDate']})")
        
        if len(games_needing_data) > 10:
            print(f"  ... and {len(games_needing_data) - 10} more")
    else:
        print("\n✅ All existing games have complete statistics!")
    
    print("="*50)
    
    return list(games_needing_data)

# Check for existing games missing statistics
existing_games_missing_stats = find_games_missing_stats()



CHECKING FOR MISSING STATISTICS

Total games in Games.csv: 72176
Games with player statistics: 72176
Games with team statistics: 72176

Games missing player stats: 0
Games missing team stats: 0
Total games needing data: 0

✅ All existing games have complete statistics!


## Fetch Detailed Box Scores

In [8]:
def get_box_score(game_id):
    """
    Fetch detailed box score for a specific game using V3 endpoint.
    """
    try:
        # Format game_id as 10-digit string with leading zeros
        game_id_str = str(game_id).strip()
        game_id_clean = ''.join(filter(str.isdigit, game_id_str))
        game_id_formatted = game_id_clean.zfill(10)
        
        time.sleep(API_DELAY)  # Rate limiting
        
        # Fetch with V3 endpoint
        boxscore = boxscoretraditionalv3.BoxScoreTraditionalV3(
            game_id=game_id_formatted,
            start_period=0,
            end_period=10,
            start_range=0,
            end_range=0,
            range_type=0
        )
        
        data_frames = boxscore.get_data_frames()
        
        if len(data_frames) < 1:
            return pd.DataFrame(), pd.DataFrame()
        
        player_stats = data_frames[0].copy()
        team_stats = data_frames[1].copy() if len(data_frames) >= 2 else pd.DataFrame()
        
        if len(player_stats) == 0:
            return pd.DataFrame(), pd.DataFrame()
        
        # Strip leading zeros from gameId for consistent matching with CSV files
        if 'gameId' in player_stats.columns:
            player_stats['gameId'] = player_stats['gameId'].astype(str).str.lstrip('0')
        if len(team_stats) > 0 and 'gameId' in team_stats.columns:
            team_stats['gameId'] = team_stats['gameId'].astype(str).str.lstrip('0')
        
        # V3 already returns correct column names - no mapping needed!
        # Combine firstName and familyName into a single PLAYER_NAME for compatibility
        if 'firstName' in player_stats.columns and 'familyName' in player_stats.columns:
            player_stats['PLAYER_NAME'] = player_stats['firstName'] + ' ' + player_stats['familyName']
        
        return player_stats, team_stats
        
    except Exception as e:
        print(f"\n  ⚠️  Error fetching game {game_id}: {str(e)}")
        return pd.DataFrame(), pd.DataFrame()

def fetch_all_box_scores(game_ids):
    """
    Fetch box scores for all games in the list using V3 endpoint.
    """
    if len(game_ids) == 0:
        return pd.DataFrame(), pd.DataFrame()
    
    # Convert all game IDs to 10-digit format
    game_ids_formatted = []
    game_id_mapping = {}
    
    for gid in game_ids:
        gid_str = str(gid).strip()
        gid_clean = ''.join(filter(str.isdigit, gid_str))
        gid_formatted = gid_clean.zfill(10)
        game_ids_formatted.append(gid_formatted)
        game_id_mapping[gid_formatted] = gid_clean
    
    print(f"\nFetching box scores for {len(game_ids_formatted)} games...")
    print(f"Estimated time: ~{(len(game_ids_formatted) * API_DELAY) / 60:.1f} minutes")
    
    all_player_stats = []
    all_team_stats = []
    no_data_games = []
    success_count = 0
    
    for i, game_id in enumerate(game_ids_formatted, 1):
        original_id = game_id_mapping[game_id]
        print(f"[{i}/{len(game_ids_formatted)}] Game {original_id}...", end=' ')
        player_stats, team_stats = get_box_score(game_id)
        
        if len(player_stats) > 0:
            all_player_stats.append(player_stats)
            success_count += 1
            print(f"✓ ({len(player_stats)} players)")
        else:
            no_data_games.append(original_id)
            print("⚠️ No data")
            
        if len(team_stats) > 0:
            all_team_stats.append(team_stats)
    
    print(f"\n{'='*50}")
    print(f"Fetched: {success_count}/{len(game_ids_formatted)} games")
    if no_data_games:
        print(f"No data: {len(no_data_games)} games (not played yet or unavailable)")
    print(f"{'='*50}")
    
    if all_player_stats:
        combined_player_stats = pd.concat(all_player_stats, ignore_index=True)
        print(f"✓ {len(combined_player_stats)} player statistics from {len(combined_player_stats['gameId'].unique())} games")
    else:
        combined_player_stats = pd.DataFrame()
        print("⚠️  No player stats were fetched")
    
    if all_team_stats:
        combined_team_stats = pd.concat(all_team_stats, ignore_index=True)
        print(f"✓ {len(combined_team_stats)} team statistics from {len(combined_team_stats['gameId'].unique())} games")
    else:
        combined_team_stats = pd.DataFrame()
        print("⚠️  No team stats were fetched")
    
    print(f"{'='*50}\n")
    
    return combined_player_stats, combined_team_stats

# Combine new games with existing games that need statistics
games_to_fetch = []

print("\n" + "="*50)
print("PREPARING TO FETCH BOX SCORES")
print("="*50)

if len(new_games_df) > 0:
    new_game_ids = new_games_df['gameId'].tolist()
    games_to_fetch.extend(new_game_ids)
    print(f"➕ New games: {len(new_games_df)}")

if len(existing_games_missing_stats) > 0:
    games_to_fetch.extend(existing_games_missing_stats)
    print(f"➕ Games with missing stats: {len(existing_games_missing_stats)}")

# Remove duplicates
games_to_fetch = list(set(games_to_fetch))

print(f"📊 Total games to fetch: {len(games_to_fetch)}")
print("="*50)

if len(games_to_fetch) > 0:
    new_player_stats_raw, new_team_stats_raw = fetch_all_box_scores(games_to_fetch)
else:
    print("\n✅ No box scores need to be fetched!")
    new_player_stats_raw = pd.DataFrame()
    new_team_stats_raw = pd.DataFrame()



PREPARING TO FETCH BOX SCORES
➕ New games: 156
📊 Total games to fetch: 156

Fetching box scores for 156 games...
Estimated time: ~1.6 minutes
[1/156] Game 0022500277... ✓ (26 players)
[2/156] Game 0022500334... ✓ (26 players)
[3/156] Game 0022500313... ✓ (27 players)
[4/156] Game 0022500076... ✓ (25 players)
[5/156] Game 0022500328... ✓ (24 players)
[6/156] Game 0022500048... ✓ (25 players)
[7/156] Game 0022501205... ✓ (25 players)
[8/156] Game 0022500353... ✓ (26 players)
[9/156] Game 0022501219... ✓ (25 players)
[10/156] Game 0022500309... ✓ (25 players)
[11/156] Game 0022500291... ✓ (23 players)
[12/156] Game 0022500061... ✓ (24 players)
[13/156] Game 0022500064... ✓ (27 players)
[14/156] Game 0022500286... ✓ (28 players)
[15/156] Game 0022500337... ✓ (27 players)
[16/156] Game 0022500318... ✓ (25 players)
[17/156] Game 0022501229... ✓ (27 players)
[18/156] Game 0022500316... ✓ (25 players)
[19/156] Game 0022500275... ✓ (24 players)
[20/156] Game 0022501218... ✓ (23 players)
[21/15

In [9]:
new_player_stats_raw

,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,...,reboundsDefensive,reboundsTotal,assists,steals,blocks,turnovers,foulsPersonal,points,plusMinusPoints,PLAYER_NAME
0,22500277,1610612738,Boston,Celtics,BOS,celtics,1627759,Jaylen,Brown,J. Brown,...,3,4,8,1,0,1,3,35,12.0,Jaylen Brown
1,22500277,1610612738,Boston,Celtics,BOS,celtics,1641775,Jordan,Walsh,J. Walsh,...,2,3,2,0,1,2,2,5,-6.0,Jordan Walsh
2,22500277,1610612738,Boston,Celtics,BOS,celtics,1629674,Neemias,Queta,N. Queta,...,1,2,0,0,0,0,2,6,-2.0,Neemias Queta
3,22500277,1610612738,Boston,Celtics,BOS,celtics,1630202,Payton,Pritchard,P. Pritchard,...,3,5,8,2,0,1,1,19,15.0,Payton Pritchard
4,22500277,1610612738,Boston,Celtics,BOS,celtics,1628401,Derrick,White,D. White,...,5,7,5,2,3,1,2,16,-2.0,Derrick White
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4004,22500351,1610612737,Atlanta,Hawks,ATL,hawks,1628379,Luke,Kennard,L. Kennard,...,3,3,2,2,0,1,0,15,10.0,Luke Kennard
4005,22500351,1610612737,Atlanta,Hawks,ATL,hawks,1630811,Keaton,Wallace,K. Wallace,...,0,0,1,0,0,0,2,0,-4.0,Keaton Wallace
4006,22500351,1610612737,Atlanta,Hawks,ATL,hawks,1642854,Asa,Newell,A. Newell,...,0,3,3,0,0,0,2,11,15.0,Asa Newell
4007,22500351,1610612737,Atlanta,Hawks,ATL,hawks,204001,Kristaps,Porziņģis,K. Porziņģis,...,0,0,0,0,0,0,0,0,0.0,Kristaps Porziņģis


## Process Player Statistics

In [10]:
def process_player_stats(raw_stats, games_df):
    """
    Convert raw player statistics to match the PlayerStatistics.csv schema.
    """
    if len(raw_stats) == 0:
        return pd.DataFrame()
    
    print("\nProcessing player statistics...")
    
    # Create a copy of games_df and ensure gameDate is in string format for matching
    games_working = games_df.copy()
    
    # Normalize game IDs (remove leading zeros) for consistent matching
    games_working['gameId'] = games_working['gameId'].astype(str).str.lstrip('0')
    
    # Convert datetime back to string format if needed for consistent dictionary keys
    if pd.api.types.is_datetime64_any_dtype(games_working['gameDate']):
        games_working['gameDate_str'] = games_working['gameDate'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    else:
        games_working['gameDate_str'] = games_working['gameDate']
    
    # Create a mapping of game_id to game info
    game_info = games_working.set_index('gameId')[['gameDate_str', 'hometeamCity', 'hometeamName', 
                                                     'awayteamCity', 'awayteamName', 'hometeamId', 
                                                     'awayteamId', 'winner']].to_dict('index')
    
    processed_stats = []
    
    for _, row in raw_stats.iterrows():
        game_id = row['gameId']  # V3 uses 'gameId', not 'GAME_ID'
        team_id = row['teamId']  # V3 uses 'teamId', not 'TEAM_ID'
        
        if game_id not in game_info:
            continue
        
        game = game_info[game_id]
        
        # Determine if this is home or away team
        is_home = 1 if team_id == game['hometeamId'] else 0
        
        if is_home:
            player_team_city = game['hometeamCity']
            player_team_name = game['hometeamName']
            opponent_city = game['awayteamCity']
            opponent_name = game['awayteamName']
        else:
            player_team_city = game['awayteamCity']
            player_team_name = game['awayteamName']
            opponent_city = game['hometeamCity']
            opponent_name = game['hometeamName']
        
        # Determine win
        win = 1 if team_id == game['winner'] else 0
        
        # V3 provides firstName and familyName separately, or use PLAYER_NAME we created
        if 'PLAYER_NAME' in row and pd.notna(row['PLAYER_NAME']):
            name_parts = row['PLAYER_NAME'].split(' ', 1)
            first_name = name_parts[0] if len(name_parts) > 0 else ''
            last_name = name_parts[1] if len(name_parts) > 1 else ''
        else:
            first_name = row['firstName'] if pd.notna(row['firstName']) else ''
            last_name = row['familyName'] if pd.notna(row['familyName']) else ''
        
        stat_record = {
            'firstName': first_name,
            'lastName': last_name,
            'personId': row['personId'],  # V3 column name
            'gameId': game_id,
            'gameDate': game['gameDate_str'],
            'playerteamCity': player_team_city,
            'playerteamName': player_team_name,
            'opponentteamCity': opponent_city,
            'opponentteamName': opponent_name,
            'gameType': '',
            'gameLabel': '',
            'gameSubLabel': '',
            'seriesGameNumber': '',
            'win': win,
            'home': is_home,
            'numMinutes': row['minutes'] if pd.notna(row['minutes']) else 0,  # V3: 'minutes'
            'points': row['points'] if pd.notna(row['points']) else 0,  # V3: 'points'
            'assists': row['assists'] if pd.notna(row['assists']) else 0,  # V3: 'assists'
            'blocks': row['blocks'] if pd.notna(row['blocks']) else 0,  # V3: 'blocks'
            'steals': row['steals'] if pd.notna(row['steals']) else 0,  # V3: 'steals'
            'fieldGoalsAttempted': row['fieldGoalsAttempted'] if pd.notna(row['fieldGoalsAttempted']) else 0,
            'fieldGoalsMade': row['fieldGoalsMade'] if pd.notna(row['fieldGoalsMade']) else 0,
            'fieldGoalsPercentage': row['fieldGoalsPercentage'] if pd.notna(row['fieldGoalsPercentage']) else 0,
            'threePointersAttempted': row['threePointersAttempted'] if pd.notna(row['threePointersAttempted']) else 0,
            'threePointersMade': row['threePointersMade'] if pd.notna(row['threePointersMade']) else 0,
            'threePointersPercentage': row['threePointersPercentage'] if pd.notna(row['threePointersPercentage']) else 0,
            'freeThrowsAttempted': row['freeThrowsAttempted'] if pd.notna(row['freeThrowsAttempted']) else 0,
            'freeThrowsMade': row['freeThrowsMade'] if pd.notna(row['freeThrowsMade']) else 0,
            'freeThrowsPercentage': row['freeThrowsPercentage'] if pd.notna(row['freeThrowsPercentage']) else 0,
            'reboundsDefensive': row['reboundsDefensive'] if pd.notna(row['reboundsDefensive']) else 0,
            'reboundsOffensive': row['reboundsOffensive'] if pd.notna(row['reboundsOffensive']) else 0,
            'reboundsTotal': row['reboundsTotal'] if pd.notna(row['reboundsTotal']) else 0,
            'foulsPersonal': row['foulsPersonal'] if pd.notna(row['foulsPersonal']) else 0,
            'turnovers': row['turnovers'] if pd.notna(row['turnovers']) else 0,
            'plusMinusPoints': row['plusMinusPoints'] if pd.notna(row['plusMinusPoints']) else 0
        }
        
        processed_stats.append(stat_record)
    
    new_stats_df = pd.DataFrame(processed_stats)
    print(f"✓ Processed {len(new_stats_df)} player statistics records")
    
    return new_stats_df

# Process player statistics
if len(new_player_stats_raw) > 0:
    # Combine old and new games for processing
    all_games_for_processing = pd.concat([games_df, new_games_df], ignore_index=True) if len(new_games_df) > 0 else games_df
    new_player_stats_df = process_player_stats(new_player_stats_raw, all_games_for_processing)
else:
    new_player_stats_df = pd.DataFrame()



Processing player statistics...
✓ Processed 4009 player statistics records


In [11]:
new_team_stats_raw

,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,minutes,fieldGoalsMade,fieldGoalsAttempted,fieldGoalsPercentage,...,reboundsOffensive,reboundsDefensive,reboundsTotal,assists,steals,blocks,turnovers,foulsPersonal,points,startersBench
0,22500277,1610612738,Boston,Celtics,BOS,celtics,137:27,32,59,0.542,...,7,14,21,23,5,4,5,10,81,Starters
1,22500277,1610612738,Boston,Celtics,BOS,celtics,137:27,21,29,0.724,...,0,16,16,6,1,4,0,11,57,Bench
2,22500277,1610612753,Orlando,Magic,ORL,magic,120:15,22,47,0.468,...,9,10,19,18,2,1,5,8,63,Starters
3,22500277,1610612753,Orlando,Magic,ORL,magic,120:15,25,53,0.472,...,9,16,25,15,1,3,2,9,66,Bench
4,22500334,1610612764,Washington,Wizards,WAS,wizards,128:14,22,43,0.512,...,5,11,16,11,3,1,11,12,60,Starters
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619,22500278,1610612746,LA,Clippers,LAC,clippers,144:49,7,18,0.389,...,2,8,10,9,1,0,5,5,19,Bench
620,22500351,1610612764,Washington,Wizards,WAS,wizards,160:25,28,48,0.583,...,2,14,16,24,6,5,13,14,79,Starters
621,22500351,1610612764,Washington,Wizards,WAS,wizards,160:25,13,32,0.406,...,2,8,10,5,0,0,7,4,37,Bench
622,22500351,1610612737,Atlanta,Hawks,ATL,hawks,161:47,34,72,0.472,...,12,24,36,28,8,3,11,10,87,Starters


## Process Team Statistics

In [12]:
def process_team_stats(raw_stats, games_df):
    """
    Convert raw team statistics to match the TeamStatistics.csv schema.
    """
    if len(raw_stats) == 0:
        return pd.DataFrame()
    
    print("\nProcessing team statistics...")
    
    # Create a copy of games_df and ensure gameDate is in string format for matching
    games_working = games_df.copy()
    
    # Normalize game IDs (remove leading zeros) for consistent matching
    games_working['gameId'] = games_working['gameId'].astype(str).str.lstrip('0')
    
    # Convert datetime back to string format if needed for consistent dictionary keys
    if pd.api.types.is_datetime64_any_dtype(games_working['gameDate']):
        games_working['gameDate_str'] = games_working['gameDate'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    else:
        games_working['gameDate_str'] = games_working['gameDate']
    
    # Create a mapping of game_id to game info
    game_info = games_working.set_index('gameId')[['gameDate_str', 'hometeamCity', 'hometeamName', 
                                                     'awayteamCity', 'awayteamName', 'hometeamId', 
                                                     'awayteamId', 'homeScore', 'awayScore', 'winner']].to_dict('index')
    
    processed_stats = []
    
    for _, row in raw_stats.iterrows():
        game_id = row['gameId']  # V3 uses 'gameId'
        team_id = row['teamId']  # V3 uses 'teamId'
        
        if game_id not in game_info:
            continue
        
        game = game_info[game_id]
        
        # Determine if this is home or away team
        is_home = 1 if team_id == game['hometeamId'] else 0
        
        if is_home:
            team_city = game['hometeamCity']
            team_name = game['hometeamName']
            opponent_city = game['awayteamCity']
            opponent_name = game['awayteamName']
            opponent_id = game['awayteamId']
            team_score = game['homeScore']
            opponent_score = game['awayScore']
        else:
            team_city = game['awayteamCity']
            team_name = game['awayteamName']
            opponent_city = game['hometeamCity']
            opponent_name = game['hometeamName']
            opponent_id = game['hometeamId']
            team_score = game['awayScore']
            opponent_score = game['homeScore']
        
        # Determine win
        win = 1 if team_id == game['winner'] else 0
        
        stat_record = {
            'gameId': game_id,
            'gameDate': game['gameDate_str'],
            'teamCity': team_city,
            'teamName': team_name,
            'teamId': team_id,
            'opponentTeamCity': opponent_city,
            'opponentTeamName': opponent_name,
            'opponentTeamId': opponent_id,
            'home': is_home,
            'win': win,
            'teamScore': team_score,
            'opponentScore': opponent_score,
            'assists': row['assists'] if pd.notna(row['assists']) else '',  # V3 column names
            'blocks': row['blocks'] if pd.notna(row['blocks']) else '',
            'steals': row['steals'] if pd.notna(row['steals']) else '',
            'fieldGoalsAttempted': row['fieldGoalsAttempted'] if pd.notna(row['fieldGoalsAttempted']) else '',
            'fieldGoalsMade': row['fieldGoalsMade'] if pd.notna(row['fieldGoalsMade']) else '',
            'fieldGoalsPercentage': row['fieldGoalsPercentage'] if pd.notna(row['fieldGoalsPercentage']) else '',
            'threePointersAttempted': row['threePointersAttempted'] if pd.notna(row['threePointersAttempted']) else '',
            'threePointersMade': row['threePointersMade'] if pd.notna(row['threePointersMade']) else '',
            'threePointersPercentage': row['threePointersPercentage'] if pd.notna(row['threePointersPercentage']) else '',
            'freeThrowsAttempted': row['freeThrowsAttempted'] if pd.notna(row['freeThrowsAttempted']) else '',
            'freeThrowsMade': row['freeThrowsMade'] if pd.notna(row['freeThrowsMade']) else '',
            'freeThrowsPercentage': row['freeThrowsPercentage'] if pd.notna(row['freeThrowsPercentage']) else '',
            'reboundsDefensive': row['reboundsDefensive'] if pd.notna(row['reboundsDefensive']) else '',
            'reboundsOffensive': row['reboundsOffensive'] if pd.notna(row['reboundsOffensive']) else '',
            'reboundsTotal': row['reboundsTotal'] if pd.notna(row['reboundsTotal']) else '',
            'foulsPersonal': row['foulsPersonal'] if pd.notna(row['foulsPersonal']) else '',
            'turnovers': row['turnovers'] if pd.notna(row['turnovers']) else '',
            'plusMinusPoints': row['plusMinusPoints'] if 'plusMinusPoints' in row and pd.notna(row['plusMinusPoints']) else '',
            'numMinutes': row['minutes'] if pd.notna(row['minutes']) else 240.0,  # V3: 'minutes'
            'q1Points': '',
            'q2Points': '',
            'q3Points': '',
            'q4Points': '',
            'benchPoints': '',
            'biggestLead': '',
            'biggestScoringRun': '',
            'leadChanges': '',
            'pointsFastBreak': '',
            'pointsFromTurnovers': '',
            'pointsInThePaint': '',
            'pointsSecondChance': '',
            'timesTied': '',
            'timeoutsRemaining': '',
            'seasonWins': '',
            'seasonLosses': '',
            'coachId': ''
        }
        
        processed_stats.append(stat_record)
    
    new_stats_df = pd.DataFrame(processed_stats)
    print(f"✓ Processed {len(new_stats_df)} team statistics records")
    
    return new_stats_df

# Process team statistics
if len(new_team_stats_raw) > 0:
    # Combine old and new games for processing
    all_games_for_processing = pd.concat([games_df, new_games_df], ignore_index=True) if len(new_games_df) > 0 else games_df
    new_team_stats_df = process_team_stats(new_team_stats_raw, all_games_for_processing)
else:
    new_team_stats_df = pd.DataFrame()



Processing team statistics...
✓ Processed 624 team statistics records


## Update Players List

In [13]:
def update_players_list(existing_players_df, new_player_stats_df):
    """
    Add any new players found in the statistics to the Players.csv file.
    """
    if len(new_player_stats_df) == 0:
        return pd.DataFrame()
    
    print("\nChecking for new players...")
    
    # Get unique players from new stats
    new_players = new_player_stats_df[['personId', 'firstName', 'lastName']].drop_duplicates()
    
    # Filter out players that already exist
    existing_player_ids = set(existing_players_df['personId'].values)
    truly_new_players = new_players[~new_players['personId'].isin(existing_player_ids)]
    
    if len(truly_new_players) == 0:
        print("No new players to add")
        return pd.DataFrame()
    
    # Create records matching Players.csv schema
    new_player_records = []
    for _, player in truly_new_players.iterrows():
        record = {
            'personId': player['personId'],
            'firstName': player['firstName'],
            'lastName': player['lastName'],
            'birthdate': '',
            'lastAttended': '',
            'country': '',
            'height': '',
            'bodyWeight': '',
            'guard': '',
            'forward': '',
            'center': '',
            'draftYear': '',
            'draftRound': '',
            'draftNumber': ''
        }
        new_player_records.append(record)
    
    new_players_df = pd.DataFrame(new_player_records)
    print(f"✓ Found {len(new_players_df)} new players to add")
    
    return new_players_df

# Update players list
if len(new_player_stats_df) > 0:
    new_players_to_add = update_players_list(players_df, new_player_stats_df)
else:
    new_players_to_add = pd.DataFrame()


Checking for new players...
✓ Found 1 new players to add


## Save Updated Data

In [14]:
def save_updated_data():
    """
    Save all updated dataframes to CSV files.
    """
    updates_made = False
    
    print("\n" + "="*50)
    print("SAVING UPDATES")
    print("="*50)
    
    # Update Games
    if len(new_games_df) > 0:
        updated_games = pd.concat([games_df, new_games_df], ignore_index=True)
        updated_games.to_csv(GAMES_FILE, index=False)
        print(f"✓ Games.csv updated: Added {len(new_games_df)} new games")
        updates_made = True
    else:
        print("○ Games.csv: No updates needed")
    
    # Update Player Statistics
    if len(new_player_stats_df) > 0:
        # Remove any existing stats for games we're updating (to avoid duplicates)
        games_being_updated = set(new_player_stats_df['gameId'].unique())
        existing_stats_to_keep = player_stats_df[~player_stats_df['gameId'].isin(games_being_updated)]
        
        # Combine kept stats with new stats
        updated_player_stats = pd.concat([existing_stats_to_keep, new_player_stats_df], ignore_index=True)
        updated_player_stats.to_csv(PLAYER_STATS_FILE, index=False)
        
        removed_count = len(player_stats_df) - len(existing_stats_to_keep)
        if removed_count > 0:
            print(f"✓ PlayerStatistics.csv updated: Replaced {removed_count} records, added {len(new_player_stats_df)} new records")
        else:
            print(f"✓ PlayerStatistics.csv updated: Added {len(new_player_stats_df)} new records")
        updates_made = True
    else:
        print("○ PlayerStatistics.csv: No updates needed")
    
    # Update Team Statistics
    if len(new_team_stats_df) > 0:
        # Remove any existing stats for games we're updating (to avoid duplicates)
        games_being_updated = set(new_team_stats_df['gameId'].unique())
        existing_stats_to_keep = team_stats_df[~team_stats_df['gameId'].isin(games_being_updated)]
        
        # Combine kept stats with new stats
        updated_team_stats = pd.concat([existing_stats_to_keep, new_team_stats_df], ignore_index=True)
        updated_team_stats.to_csv(TEAM_STATS_FILE, index=False)
        
        removed_count = len(team_stats_df) - len(existing_stats_to_keep)
        if removed_count > 0:
            print(f"✓ TeamStatistics.csv updated: Replaced {removed_count} records, added {len(new_team_stats_df)} new records")
        else:
            print(f"✓ TeamStatistics.csv updated: Added {len(new_team_stats_df)} new records")
        updates_made = True
    else:
        print("○ TeamStatistics.csv: No updates needed")
    
    # Update Players
    if len(new_players_to_add) > 0:
        updated_players = pd.concat([players_df, new_players_to_add], ignore_index=True)
        updated_players.to_csv(PLAYERS_FILE, index=False)
        print(f"✓ Players.csv updated: Added {len(new_players_to_add)} new players")
        updates_made = True
    else:
        print("○ Players.csv: No updates needed")
    
    print("="*50)
    
    if updates_made:
        print("\n✅ Dataset successfully updated with new data!")
    else:
        print("\n✅ Dataset is already up to date!")
    
    return updates_made

# Save all updates
save_updated_data()


SAVING UPDATES
✓ Games.csv updated: Added 156 new games
✓ PlayerStatistics.csv updated: Added 4009 new records
✓ TeamStatistics.csv updated: Added 624 new records
✓ Players.csv updated: Added 1 new players

✅ Dataset successfully updated with new data!


True

## Validate Data Integrity

In [15]:
def validate_final_data_integrity():
    """
    Final check to verify all games now have corresponding player and team statistics.
    """
    print("\n" + "="*50)
    print("FINAL DATA INTEGRITY CHECK")
    print("="*50)
    
    # Load the updated data if changes were made
    if len(new_games_df) > 0 or len(new_player_stats_df) > 0:
        print("\nReloading data to verify updates...")
        final_games_df = pd.read_csv(GAMES_FILE)
        final_player_stats_df = pd.read_csv(PLAYER_STATS_FILE)
        final_team_stats_df = pd.read_csv(TEAM_STATS_FILE)
    else:
        final_games_df = games_df
        final_player_stats_df = player_stats_df
        final_team_stats_df = team_stats_df
    
    # Get unique game IDs from each dataset
    games_with_data = set(final_games_df['gameId'].unique())
    games_with_player_stats = set(final_player_stats_df['gameId'].unique())
    games_with_team_stats = set(final_team_stats_df['gameId'].unique())
    
    # Find games missing statistics
    games_missing_player_stats = games_with_data - games_with_player_stats
    games_missing_team_stats = games_with_data - games_with_team_stats
    
    print(f"\nTotal games in dataset: {len(games_with_data)}")
    print(f"Games with player statistics: {len(games_with_player_stats)}")
    print(f"Games with team statistics: {len(games_with_team_stats)}")
    
    if games_missing_player_stats:
        print(f"\n⚠️  WARNING: {len(games_missing_player_stats)} games are still missing player statistics!")
        missing_games_df = final_games_df[final_games_df['gameId'].isin(games_missing_player_stats)].sort_values('gameDate', ascending=False)
        print("\nGames still missing player statistics:")
        for idx, game in missing_games_df.head(10).iterrows():
            print(f"  - {game['gameId']}: {game['awayteamCity']} {game['awayteamName']} @ {game['hometeamCity']} {game['hometeamName']}")
        if len(games_missing_player_stats) > 10:
            print(f"  ... and {len(games_missing_player_stats) - 10} more")
    else:
        print("\n✅ All games have player statistics!")
    
    if games_missing_team_stats:
        print(f"\n⚠️  WARNING: {len(games_missing_team_stats)} games are still missing team statistics!")
    else:
        print("✅ All games have team statistics!")
    
    print("="*50)
    
    return len(games_missing_player_stats) == 0 and len(games_missing_team_stats) == 0

# Run final validation
validation_passed = validate_final_data_integrity()


FINAL DATA INTEGRITY CHECK

Reloading data to verify updates...

Total games in dataset: 72332
Games with player statistics: 72332
Games with team statistics: 72332

✅ All games have player statistics!
✅ All games have team statistics!


## Summary

In [16]:
print("\n" + "="*50)
print("UPDATE SUMMARY")
print("="*50)
print(f"New games added: {len(new_games_df) if len(new_games_df) > 0 else 0}")
print(f"Existing games with missing stats: {len(existing_games_missing_stats) if len(existing_games_missing_stats) > 0 else 0}")
print(f"Total box scores fetched: {len(games_to_fetch) if len(games_to_fetch) > 0 else 0}")
print(f"Player statistics added/updated: {len(new_player_stats_df) if len(new_player_stats_df) > 0 else 0}")
print(f"Team statistics added/updated: {len(new_team_stats_df) if len(new_team_stats_df) > 0 else 0}")
print(f"New players added: {len(new_players_to_add) if len(new_players_to_add) > 0 else 0}")
print("="*50)
print(f"\nDataset now current through: {datetime.now().strftime('%Y-%m-%d')}")

if validation_passed:
    print("\n✅ Update complete! All games have statistics. You can run this notebook again tomorrow to get new data.")
else:
    print("\n⚠️  Update complete, but some games are still missing statistics. Check the validation output above.")


UPDATE SUMMARY
New games added: 156
Existing games with missing stats: 0
Total box scores fetched: 156
Player statistics added/updated: 4009
Team statistics added/updated: 624
New players added: 1

Dataset now current through: 2025-12-15

✅ Update complete! All games have statistics. You can run this notebook again tomorrow to get new data.
